In [10]:
## install finrl library
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

# 导入所需库

In [11]:
# 导入必要的库
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from stable_baselines3 import A2C, DDPG, PPO, TD3, SAC
import torch
import time

from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR
from finrl.main import check_and_make_directories
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号
plt.rcParams["figure.figsize"] = (15,7)       # 设置图表大小

# 是否启用GPU

In [12]:
# 检查GPU可用性
print("检查GPU可用性...")
use_cuda = torch.cuda.is_available()
if use_cuda:
    cuda_device_count = torch.cuda.device_count()
    cuda_device_name = torch.cuda.get_device_name(0)
    print(f"✓ 发现 {cuda_device_count} 个可用的GPU设备")
    print(f"✓ 当前使用: {cuda_device_name}")
else:
    print("✗ 未发现可用的GPU，将使用CPU进行回测")

# 确保目录存在
check_and_make_directories([TRAINED_MODEL_DIR, 'results'])

检查GPU可用性...
✓ 发现 1 个可用的GPU设备
✓ 当前使用: NVIDIA GeForce RTX 3060 Laptop GPU


# 读取数据

In [13]:
# Part 2. 回测准备

# 加载数据（保持路径不变）
train = pd.read_csv("data/processed_data/train_data_20150101~20250101.csv")
trade = pd.read_csv("data/processed_data/test_data_20150101~20250101.csv")

# 设置索引格式 - 与示例代码保持一致
train = train.set_index(train.columns[0])
train.index.names = [""]
trade = trade.set_index(trade.columns[0])
trade.index.names = [""]

print(f"训练数据共 {len(train)} 条记录")
print(f"测试数据共 {len(trade)} 条记录")

训练数据共 950608 条记录
测试数据共 236472 条记录


# 回测模型选择

In [14]:
# 设置要使用的模型
if_using_a2c = True
if_using_ddpg = True
if_using_ppo = True
if_using_td3 = True
if_using_sac = True

# 加载已训练的模型 并指定哪些需要用cpu 哪些用gpu
trained_a2c = (
    A2C.load(TRAINED_MODEL_DIR + "/best/a2c/best_model", device="cuda")
    if if_using_a2c
    else None
)
trained_ddpg = (
    DDPG.load(TRAINED_MODEL_DIR + "/best/ddpg/best_model", device="cuda")
    if if_using_ddpg
    else None
)
trained_ppo = (
    PPO.load(TRAINED_MODEL_DIR + "/best/ppo/best_model", device="cuda")
    if if_using_ppo
    else None
)
trained_td3 = (
    TD3.load(TRAINED_MODEL_DIR + "/best/td3/best_model", device="cuda")
    if if_using_td3
    else None
)
trained_sac = (
    SAC.load(TRAINED_MODEL_DIR + "/best/sac/best_model", device="cuda")
    if if_using_sac
    else None
)

e:\DevelopentTools\Anaconda3\envs\FinRL\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
e:\DevelopentTools\Anaconda3\envs\FinRL\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `exp

# 设置回测参数

In [15]:
def reduce_stock_dataset_consistent(df, top_n_stocks=15):
    """确保使用与训练时相同的股票集合"""
    print(f"回测前股票数量: {len(df.tic.unique())}")

    # 使用与训练时相同的选股逻辑
    stock_volume = df.groupby("tic")["volume"].mean().sort_values(ascending=False)
    selected_stocks = stock_volume.head(top_n_stocks).index.tolist()

    # 过滤数据
    reduced_df = df[df.tic.isin(selected_stocks)].copy()
    print(f"回测时股票数量: {len(reduced_df.tic.unique())}")
    return reduced_df


# 应用相同的股票选择过滤
trade = reduce_stock_dataset_consistent(trade, top_n_stocks=15)  # 必须与训练时相同
# 构建交易环境参数

stock_dimension = len(trade.tic.unique())

state_space = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension

print(f"股票维度: {stock_dimension}, 状态空间: {state_space}")


# 设置交易成本和初始持仓

buy_cost_list = sell_cost_list = [0.001] * stock_dimension

num_stock_shares = [0] * stock_dimension


env_kwargs = {

    "hmax": 100,

    "initial_amount": 1000000,

    "num_stock_shares": num_stock_shares,

    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,

    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,

    "reward_scaling": 1e-3,

}


# 构建回测环境

e_trade_gym = StockTradingEnv(

    df=trade, turbulence_threshold=70, risk_indicator_col="vix", **env_kwargs
)

回测前股票数量: 472
回测时股票数量: 15
股票维度: 15, 状态空间: 151


# 开始回测

In [16]:
# A2C模型回测
if if_using_a2c:
    print("正在回测A2C模型...")
    df_account_value_a2c, df_actions_a2c = DRLAgent.DRL_prediction(
        model=trained_a2c, environment=e_trade_gym
    )
    print(
        f"A2C回测完成，最终资产: ${df_account_value_a2c['account_value'].iloc[-1]:,.2f}"
    )
else:
    df_account_value_a2c, df_actions_a2c = None, None

# DDPG模型回测
if if_using_ddpg:
    print("正在回测DDPG模型...")
    df_account_value_ddpg, df_actions_ddpg = DRLAgent.DRL_prediction(
        model=trained_ddpg, environment=e_trade_gym
    )
    print(
        f"DDPG回测完成，最终资产: ${df_account_value_ddpg['account_value'].iloc[-1]:,.2f}"
    )
else:
    df_account_value_ddpg, df_actions_ddpg = None, None

# PPO模型回测
if if_using_ppo:
    print("正在回测PPO模型...")
    df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
        model=trained_ppo, environment=e_trade_gym
    )
    print(
        f"PPO回测完成，最终资产: ${df_account_value_ppo['account_value'].iloc[-1]:,.2f}"
    )
else:
    df_account_value_ppo, df_actions_ppo = None, None

# TD3模型回测
if if_using_td3:
    print("正在回测TD3模型...")
    df_account_value_td3, df_actions_td3 = DRLAgent.DRL_prediction(
        model=trained_td3, environment=e_trade_gym
    )
    print(
        f"TD3回测完成，最终资产: ${df_account_value_td3['account_value'].iloc[-1]:,.2f}"
    )
else:
    df_account_value_td3, df_actions_td3 = None, None

# SAC模型回测
if if_using_sac:
    print("正在回测SAC模型...")
    df_account_value_sac, df_actions_sac = DRLAgent.DRL_prediction(
        model=trained_sac, environment=e_trade_gym
    )
    print(
        f"SAC回测完成，最终资产: ${df_account_value_sac['account_value'].iloc[-1]:,.2f}"
    )
else:
    df_account_value_sac, df_actions_sac = None, None

正在回测A2C模型...
hit end!
A2C回测完成，最终资产: $1,621,196.88
正在回测DDPG模型...
hit end!
DDPG回测完成，最终资产: $2,499,344.11
正在回测PPO模型...
hit end!
PPO回测完成，最终资产: $2,630,030.25
正在回测TD3模型...
hit end!
TD3回测完成，最终资产: $1,812,045.59
正在回测SAC模型...
hit end!
SAC回测完成，最终资产: $2,681,196.69


# 添加参考基准

In [17]:
# # Part 3: 均值方差优化(MVO) - 整合版
# def mvo_backtest(
#     train_df, trade_df, initial_amount=1e6, transaction_cost=0.001, verbose=True
# ):
#     """增强版MVO回测流程"""
#     # 股票过滤（与强化学习模型保持一致）
#     selected_stocks = trade_df.tic.unique().tolist()
#     train_filtered = train_df[train_df.tic.isin(selected_stocks)]
#     trade_filtered = trade_df[trade_df.tic.isin(selected_stocks)]

#     # 准备价格矩阵
#     train_price_matrix = train_filtered.pivot(
#         index="date", columns="tic", values="close"
#     )
#     trade_price_matrix = trade_filtered.pivot(
#         index="date", columns="tic", values="close"
#     )

#     if verbose:
#         print(f"股票数量: {len(selected_stocks)}")
#         print(
#             f"训练数据时间范围: {train_price_matrix.index.min()} 至 {train_price_matrix.index.max()}"
#         )
#         print(
#             f"回测数据时间范围: {trade_price_matrix.index.min()} 至 {trade_price_matrix.index.max()}"
#         )

#     # 使用PyPortfolioOpt计算最优权重
#     from pypfopt import expected_returns, risk_models, EfficientFrontier

#     mu = expected_returns.mean_historical_return(train_price_matrix)
#     S = risk_models.sample_cov(train_price_matrix)

#     if verbose:
#         print("\n平均历史收益率前五名股票:")
#         print(mu.sort_values(ascending=False).head(5))

#     # 创建高效前沿并计算最优夏普比率组合
#     ef = EfficientFrontier(mu, S, weight_bounds=(0, 0.5))
#     weights = ef.max_sharpe()
#     cleaned_weights = ef.clean_weights()

#     # 显示投资组合性能指标
#     if verbose:
#         expected_annual_return, annual_volatility, sharpe_ratio = (
#             ef.portfolio_performance()
#         )
#         print(f"\n投资组合预期表现:")
#         print(f"预期年化收益率: {expected_annual_return:.2%}")
#         print(f"年化波动率: {annual_volatility:.2%}")
#         print(f"夏普比率: {sharpe_ratio:.2f}")

#         # 展示权重分布
#         print("\n权重分配:")
#         weights_df = pd.Series(cleaned_weights)
#         weights_df = weights_df[weights_df > 0.01].sort_values(ascending=False)
#         for ticker, weight in weights_df.items():
#             print(f"{ticker}: {weight:.2%}")

#     # 提取权重数组，保持与价格矩阵列顺序一致
#     weights_array = np.array(
#         [cleaned_weights.get(s, 0) for s in trade_price_matrix.columns]
#     )

#     # 应用交易成本后的初始配置
#     initial_investment = initial_amount * (1 - transaction_cost)
#     if verbose:
#         print(f"\n初始投资: ${initial_amount:,.2f}")
#         print(f"交易成本: ${initial_amount * transaction_cost:,.2f}")
#         print(f"净投资: ${initial_investment:,.2f}")

#     # 计算股票份额
#     shares = (initial_investment * weights_array) / trade_price_matrix.iloc[0].values

#     # 计算每日资产价值
#     portfolio_values = (trade_price_matrix * shares).sum(axis=1)

#     # 计算回测结果指标
#     total_return = (portfolio_values.iloc[-1] / portfolio_values.iloc[0]) - 1
#     daily_returns = portfolio_values.pct_change().dropna()
#     annual_return = (1 + total_return) ** (252 / len(portfolio_values)) - 1
#     annual_vol = daily_returns.std() * np.sqrt(252)
#     sharpe = annual_return / annual_vol

#     if verbose:
#         print(f"\n回测结果:")
#         print(f"总收益率: {total_return:.2%}")
#         print(f"年化收益率: {annual_return:.2%}")
#         print(f"年化波动率: {annual_vol:.2%}")
#         print(f"夏普比率: {sharpe:.2f}")

#         # 计算最大回撤
#         cum_returns = (1 + daily_returns).cumprod()
#         peak = cum_returns.cummax()
#         drawdown = (cum_returns / peak) - 1
#         max_dd = drawdown.min()
#         print(f"最大回撤: {max_dd:.2%}")

#     return portfolio_values


# # 执行MVO回测
# print("执行均值方差优化(MVO)回测...")
# mvo_portfolio = mvo_backtest(train, trade)
# MVO_result = pd.DataFrame({"Mean Var": mvo_portfolio.values}, index=mvo_portfolio.index)

In [18]:
# Part 4: S&P 500指数作为基准

# 使用回测时间范围
TRAIN_START_DATE = "2015-01-01"
TRAIN_END_DATE = "2023-01-01"
TRADE_START_DATE = "2023-01-01"
TRADE_END_DATE = "2025-01-01"

# 获取S&P500指数数据
df_spx = YahooDownloader(
    start_date=TRADE_START_DATE, end_date=TRADE_END_DATE, ticker_list=["^GSPC"]
).fetch_data()

# 处理S&P500数据，设置初始资金一致
df_spx = df_spx[["date", "close"]]
fst_day = df_spx["close"].iloc[0]
spx = pd.merge(
    df_spx["date"],
    df_spx["close"].div(fst_day).mul(1000000),
    how="outer",
    left_index=True,
    right_index=True,
).set_index("date")

[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (502, 8)


# 整合回测结果

In [19]:
# Part 5: 回测结果整合与可视化

# 处理结果数据
df_result_a2c = (
    df_account_value_a2c.set_index(df_account_value_a2c.columns[0])
    if if_using_a2c and df_account_value_a2c is not None
    else None
)
df_result_ddpg = (
    df_account_value_ddpg.set_index(df_account_value_ddpg.columns[0])
    if if_using_ddpg and df_account_value_ddpg is not None
    else None
)
df_result_ppo = (
    df_account_value_ppo.set_index(df_account_value_ppo.columns[0])
    if if_using_ppo and df_account_value_ppo is not None
    else None
)
df_result_td3 = (
    df_account_value_td3.set_index(df_account_value_td3.columns[0])
    if if_using_td3 and df_account_value_td3 is not None
    else None
)
df_result_sac = (
    df_account_value_sac.set_index(df_account_value_sac.columns[0])
    if if_using_sac and df_account_value_sac is not None
    else None
)

# 创建结果数据框 - 与示例代码保持一致的格式
result_data = {}

if if_using_a2c and df_result_a2c is not None:
    result_data["a2c"] = df_result_a2c["account_value"]

if if_using_ddpg and df_result_ddpg is not None:
    result_data["ddpg"] = df_result_ddpg["account_value"]

if if_using_ppo and df_result_ppo is not None:
    result_data["ppo"] = df_result_ppo["account_value"]

if if_using_td3 and df_result_td3 is not None:
    result_data["td3"] = df_result_td3["account_value"]

if if_using_sac and df_result_sac is not None:
    result_data["sac"] = df_result_sac["account_value"]

# 添加基准
# result_data["mvo"] = MVO_result["Mean Var"]
result_data["spx"] = spx["close"]  # 使用S&P500

# 创建结果DataFrame
result = pd.DataFrame(result_data)

# 绘制结果对比图 - 与示例代码风格一致
plt.rcParams["figure.figsize"] = (15, 5)
plt.figure()
result.plot()
plt.title("回测结果对比")
plt.xlabel("日期")
plt.ylabel("账户价值")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("results/backtest_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

E:\TEMP\ipykernel_53100\2014003944.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
